In [1]:
import json, sys, copy, os, igl
import numpy as np
import meshio

def is_inside(v, f, p):
    winds = igl.winding_number(v, f, p)
    mask = np.array(np.rint(winds), dtype=int)
    
    return np.array([e % 2 == 1 for e in mask])

def change_file_extension(filename, new_extension):
  base, ext = os.path.splitext(filename)
  return base + "." + new_extension if ext else filename + "." + new_extension

In [2]:
import json, sys, copy, os, igl
import numpy as np
import meshio

def is_inside(v, f, p):
    winds = igl.winding_number(v, f, p)
    mask = np.array(np.rint(winds), dtype=int)
    
    return np.array([e % 2 == 1 for e in mask])

def change_file_extension(filename, new_extension):
  base, ext = os.path.splitext(filename)
  return base + "." + new_extension if ext else filename + "." + new_extension

In [3]:
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
microstructure_repo_path = notebook_dir
matopt_repo_path = os.path.join(os.path.dirname(microstructure_repo_path), "matopt")

input_surface = "Fingertip_Shell_V1.stl"  # USER: Change this to your STL file path
stitch_cells_cli = os.path.join(microstructure_repo_path, "build/isosurface_inflator/cut_cells_cli")
only_cube_cells = False

In [4]:
E = 0.05
nu = 0.3
cell_size = 6
# out_path = str(E) + "_" + str(cell_size) + ".obj"
out_path = f"{input_surface[:-4]}_{E}_{cell_size}.obj_{only_cube_cells}.obj"
print(out_path)

m = meshio.read(input_surface)
input_surface = change_file_extension(input_surface, 'obj')
m.write(input_surface)
v = m.points.astype(float)
f = m.cells[0].data
bbox = [np.amin(v, axis=0), np.amax(v, axis=0)]

# corner0 = list(map(int, np.ceil(bbox[0] / cell_size)))
corner0 = list(map(int, np.ceil(bbox[0] / cell_size) - 1))
corner1 = list(map(int, np.floor(bbox[1] / cell_size)))

print(corner0, corner1)

# use return E for uniform Young's modulus
# use i,j,k to vary the Young's modulus through the dimensions
def young(i, j, k):
    return E
    
    
    
    

Fingertip_Shell_V1_0.05_6.obj_False.obj
[-1, -3, -2] [2, 2, 1]


In [5]:
sys.path.insert(0, os.path.join(matopt_repo_path, 'tools/material2geometry'))
from material2geometry import Material2Geometry

In [6]:
mat2geo = Material2Geometry(in_path=os.path.join(matopt_repo_path, "tools/material2geometry/0646_geo_1_coeffs.txt"))

Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]


In [7]:
patterns = []
entry = {"params": [],
"symmetry": "Cubic",
"pattern": os.path.join(microstructure_repo_path, "data/patterns/3D/reference_wires/pattern0646.wire"),
"index": [0,0,0]}

for i in range(corner0[0], 1 + corner1[0]):
    for j in range(corner0[1], 1 + corner1[1]):
        for k in range(corner0[2], 1 + corner1[2]):
            geo_params = mat2geo.evaluate(nu, young(i, j, k))
            entry["params"] = geo_params
            entry["index"] = [i,j,k]
            patterns.append(copy.deepcopy(entry))

with open("data.json", 'w') as f:
    json.dump(patterns, f)

In [8]:
patterns = []
entry = {"params": [],
"symmetry": "Cubic",
"pattern": os.path.join(microstructure_repo_path, "data/patterns/3D/reference_wires/pattern0646.wire"),
"index": [0,0,0]}

for i in range(corner0[0], 1 + corner1[0]):
    for j in range(corner0[1], 1 + corner1[1]):
        for k in range(corner0[2], 1 + corner1[2]):
            geo_params = mat2geo.evaluate(nu, young(i, j, k))
            entry["params"] = geo_params
            entry["index"] = [i,j,k]
            patterns.append(copy.deepcopy(entry))

with open("data.json", 'w') as f:
    json.dump(patterns, f)

In [9]:
os.system(stitch_cells_cli + " -p data.json " +
           (("--surface " + input_surface) if not only_cube_cells else "") + 
           " --gridSize " + str(cell_size) + " -o " + out_path + " -r 50")
# print((stitch_cells_cli + " -p data.json " +
#            (("--surface " + input_surface) if not only_cube_cells else "") + 
#            " --gridSize " + str(cell_size) + " -o " + out_path + " -r 50"))

1

In [10]:
for c in [1, 2, 3, 4, 5, 6, 7, 8]:
    for E in [1e-3, 2e-3, 5e-3, 1e-2]:
        print(f"Processing for c={c} and E={E}")
        print((np.array(mat2geo.evaluate(nu, E)[5:-4]) * c).tolist())

Processing for c=1 and E=0.001
[0.12500001633203092, 0.042433818399143196, 0.10558233037085027, 0.15000001959843665]
Processing for c=1 and E=0.002
[0.1250000179168281, 0.04501351489539214, 0.108725085895553, 0.15000002150019334]
Processing for c=1 and E=0.005
[0.12500002581132189, 0.05237744112630513, 0.11774100482050118, 0.1500000309735861]
Processing for c=1 and E=0.01
[0.12500004299155912, 0.06344895227780598, 0.131437387500655, 0.15000005158987087]
Processing for c=2 and E=0.001
[0.25000003266406184, 0.08486763679828639, 0.21116466074170054, 0.3000000391968733]
Processing for c=2 and E=0.002
[0.2500000358336562, 0.09002702979078428, 0.217450171791106, 0.3000000430003867]
Processing for c=2 and E=0.005
[0.25000005162264377, 0.10475488225261026, 0.23548200964100235, 0.3000000619471722]
Processing for c=2 and E=0.01
[0.25000008598311824, 0.12689790455561195, 0.26287477500131, 0.30000010317974174]
Processing for c=3 and E=0.001
[0.37500004899609274, 0.12730145519742958, 0.316746991112

In [11]:
# To check printability of the geometry, we can create a DataFrame to summarize the minimum dimensions for different cell sizes and Young's moduli.Based on your printer nozzle size, we can determine if the geometry is printable or not.

import pandas as pd
import numpy as np

c_values = [1, 2, 3, 4, 5, 6, 7, 8, 16]
E_values = [0.05, 0.1, 0.2, 0.3, 0.5]

df = pd.DataFrame(index=c_values, columns=E_values)

df.index.name = "Cell Size (c)"
df.columns.name = "Young's Modulus (E)"

for c in c_values:
    for E in E_values:
        arr = np.array(mat2geo.evaluate(nu, E)[5:-4]) * c
        min_dim = np.min(arr)        
        if min_dim < 0.4:
            df.loc[c, E] = "Unprintable"
        else:
            df.loc[c, E] = f"{min_dim:.4f}"

df

Young's Modulus (E),0.05,0.10,0.20,0.30,0.50
Cell Size (c),,,,,
1,Unprintable,Unprintable,Unprintable,Unprintable,Unprintable
2,Unprintable,Unprintable,Unprintable,Unprintable,Unprintable
3,Unprintable,Unprintable,Unprintable,Unprintable,Unprintable
4,0.4391,0.5000,0.4998,0.4996,0.4990
5,0.5488,0.6250,0.6248,0.6245,0.6237
6,0.6586,0.7499,0.7498,0.7494,0.7484
7,0.7684,0.8749,0.8747,0.8743,0.8732
8,0.8781,0.9999,0.9997,0.9992,0.9979
16,1.7563,1.9999,1.9994,1.9985,1.9958
